In [19]:
import numpy as np
import random
import torch
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import glob
from pathlib import Path

seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Reproducibility seeds set.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Reproducibility seeds set.
Using device: cpu


In [20]:
def add_technical_indicators(df):
    df = df.copy()
    # Simple and exponential moving averages
    df['SMA_10'] = df['Close'].rolling(window=10).mean()
    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    # Relative Strength Index (RSI)
    delta = df['Close'].diff()
    up, down = delta.copy(), delta.copy()
    up[up < 0] = 0
    down[down > 0] = 0
    roll_up = up.rolling(window=14).mean()
    roll_down = down.abs().rolling(window=14).mean()
    RS = roll_up / roll_down
    df['RSI_14'] = 100.0 - (100.0 / (1.0 + RS))
    df = df.dropna()
    return df

In [21]:
# Adjust this to your local CSV folder path
csv_dir = Path("data/stocks")  

# If you want to process a single ticker:
symbol = "AAPL"  # replace with any ticker matching a CSV filename
csv_path = csv_dir / f"{symbol}.csv"
df = pd.read_csv(csv_path, parse_dates=["Date"])
df.sort_values("Date", inplace=True)
df.set_index("Date", inplace=True)

# Add technical indicators (using the function you defined earlier)
df = add_technical_indicators(df)

# Quick sanity check
print(f"Loaded {len(df)} days of data for {symbol}")
print(df.tail())

# Cell 4: Build feature & target tensors from the combined DataFrame
features = ['Open','High','Low','Close','Volume','SMA_10','EMA_10','RSI_14']
X_all = df[features].values
y_all = np.log(df['Close'].values[1:] / df['Close'].values[:-1])
X_all = X_all[:-1]  # align features & targets

# Re-create DataLoader splits (with reproducible seed)
from torch.utils.data import TensorDataset, DataLoader
X_tensor = torch.tensor(X_all, dtype=torch.float32)
y_tensor = torch.tensor(y_all, dtype=torch.float32).unsqueeze(1)
dataset = TensorDataset(X_tensor, y_tensor)

train_size = int(len(dataset) * 0.8)
val_size   = int(len(dataset) * 0.1)
test_size  = len(dataset) - train_size - val_size
train_ds, val_ds, test_ds = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(seed)
)

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)

print(f"DataLoaders ready: {train_size} train / {val_size} val / {test_size} test samples")

Loaded 9895 days of data for AAPL
                  Open        High         Low       Close   Adj Close  \
Date                                                                     
2020-03-26  246.520004  258.679993  246.360001  258.440002  258.440002   
2020-03-27  252.750000  255.869995  247.050003  247.740005  247.740005   
2020-03-30  250.740005  255.520004  249.399994  254.809998  254.809998   
2020-03-31  255.600006  262.489990  252.000000  254.289993  254.289993   
2020-04-01  246.500000  248.720001  239.130005  240.910004  240.910004   

              Volume      SMA_10      EMA_10     RSI_14  
Date                                                     
2020-03-26  63021800  246.894002  250.351306  43.065694  
2020-03-27  51054200  243.871002  249.876524  45.578425  
2020-03-30  41994100  245.131001  250.773519  42.224034  
2020-03-31  49250500  245.274001  251.412878  44.345175  
2020-04-01  43956200  244.698001  249.503265  47.885617  
DataLoaders ready: 7915 train / 989 val /

In [22]:
SEQ_LEN = 60
features = ['Open','High','Low','Close','Volume','SMA_10','EMA_10','RSI_14']

# 1) raw arrays
X_all = df[features].values                         # (N, F)
y_all = np.log(df['Close'].values[1:] / df['Close'].values[:-1])
X_all = X_all[:-1]                                  # align lengths

# 2) slide windows
X_windows, y_windows = [], []
for i in range(len(X_all) - SEQ_LEN):
    X_windows.append(X_all[i : i + SEQ_LEN])        # (SEQ_LEN, F)
    y_windows.append(y_all[i + SEQ_LEN])            # next-day return
X_windows = np.stack(X_windows)                     # (M, SEQ_LEN, F)
y_windows = np.array(y_windows)                     # (M,)

# 3) TensorDataset & splits
X_t = torch.tensor(X_windows, dtype=torch.float32)
y_t = torch.tensor(y_windows, dtype=torch.float32).unsqueeze(1)
ds  = TensorDataset(X_t, y_t)

n       = len(ds)
n_train = int(n*0.8)
n_val   = int(n*0.1)
n_test  = n - n_train - n_val
train_ds, val_ds, test_ds = torch.utils.data.random_split(
    ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(seed)
)

batch_size   = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)

print(f"Windowed data → {n} samples, splits: {n_train}/{n_val}/{n_test}")

Windowed data → 9834 samples, splits: 7867/983/984


In [23]:
class StockTransformer(nn.Module):
    def __init__(self, input_dim, emb_dim, num_heads, num_layers, ff_dim, dropout):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, emb_dim)
        self.pos_encoder = nn.Parameter(
            torch.zeros(1, 500, emb_dim)
        )  # max sequence length 500; adjust as needed
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation='relu'
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        self.output_head = nn.Linear(emb_dim, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, input_dim)
        x = self.input_proj(x)              # -> (batch, seq_len, emb_dim)
        seq_len = x.size(1)
        x = x + self.pos_encoder[:, :seq_len, :]
        x = x.permute(1, 0, 2)              # Transformer expects (seq_len, batch, emb_dim)
        x = self.transformer_encoder(x)
        x = x.permute(1, 0, 2)              # back to (batch, seq_len, emb_dim)
        # take the last time step's embedding
        out = self.output_head(x[:, -1, :])
        return out

# Instantiate the model
input_dim = len(features)  # e.g., 8 features including SMA, EMA, RSI
model = StockTransformer(
    input_dim=input_dim,
    emb_dim=64,
    num_heads=4,
    num_layers=3,
    ff_dim=128,
    dropout=0.1
).to(DEVICE)

print(model)

StockTransformer(
  (input_proj): Linear(in_features=8, out_features=64, bias=True)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (output_head): Linear(in_features=64, out_features=1, bias=True)
)


C:\Users\denis\anaconda3\lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [24]:
criterion = nn.SmoothL1Loss()  # Huber loss
optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

In [25]:
best_val_loss = float('inf')
patience = 0
max_epochs = 100

for epoch in range(1, max_epochs + 1):
    model.train()
    train_losses = []
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        out = model(Xb)
        loss = criterion(out, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    
    # Validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            out = model(Xb)
            val_losses.append(criterion(out, yb).item())
    val_loss = np.mean(val_losses)
    
    scheduler.step(val_loss)
    print(f"Epoch {epoch}: Train={np.mean(train_losses):.6f}, Val={val_loss:.6f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        patience = 0
    else:
        patience += 1
        if patience >= 15:
            print("Early stopping triggered.")
            break

Epoch 1: Train=0.005402, Val=0.000538
Epoch 2: Train=0.002053, Val=0.000441
Epoch 3: Train=0.001471, Val=0.000509
Epoch 4: Train=0.001156, Val=0.000499
Epoch 5: Train=0.001011, Val=0.000402
Epoch 6: Train=0.000856, Val=0.000420
Epoch 7: Train=0.000825, Val=0.000421
Epoch 8: Train=0.000743, Val=0.000893
Epoch 9: Train=0.000724, Val=0.000412
Epoch 10: Train=0.000658, Val=0.000482
Epoch 11: Train=0.000666, Val=0.000413
Epoch 12: Train=0.000613, Val=0.000437
Epoch 13: Train=0.000623, Val=0.000406
Epoch 14: Train=0.000606, Val=0.000419
Epoch 15: Train=0.000599, Val=0.000533
Epoch 16: Train=0.000589, Val=0.000401
Epoch 17: Train=0.000575, Val=0.000402
Epoch 18: Train=0.000567, Val=0.000400
Epoch 19: Train=0.000570, Val=0.000407
Epoch 20: Train=0.000582, Val=0.000415
Epoch 21: Train=0.000559, Val=0.000415
Epoch 22: Train=0.000541, Val=0.000399
Epoch 23: Train=0.000542, Val=0.000431
Epoch 24: Train=0.000533, Val=0.000407
Epoch 25: Train=0.000530, Val=0.000529
Epoch 26: Train=0.000550, Val=0.00

In [26]:
features = ['Open','High','Low','Close','Volume','SMA_10','EMA_10','RSI_14']
close_idx = features.index('Close')  # index of the 'Close' feature

model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_preds, all_actuals, all_last_closes = [], [], []

with torch.no_grad():
    for Xb, yb in test_loader:
        # Xb shape: (batch, seq_len, num_feats)
        Xb = Xb.to(DEVICE)
        yb = yb.to(DEVICE)
        
        # 1. Predict log-returns
        out = model(Xb)
        preds_batch = out.cpu().numpy().flatten()
        actuals_batch = yb.cpu().numpy().flatten()
        
        # 2. Extract the last-close price of each window
        #    Xb[:, -1, close_idx] is the true close at time t-1
        last_closes_batch = Xb[:, -1, close_idx].cpu().numpy()
        
        # 3. Accumulate
        all_preds.extend(preds_batch)
        all_actuals.extend(actuals_batch)
        all_last_closes.extend(last_closes_batch)

# Convert to arrays
all_preds = np.array(all_preds)
all_actuals = np.array(all_actuals)
all_last_closes = np.array(all_last_closes)

# Compute price-level predictions and true prices
pred_prices = all_last_closes * np.exp(all_preds)
true_prices = all_last_closes * np.exp(all_actuals)

# Compute price-level MAPE
price_errors = pred_prices - true_prices
price_mape = np.mean(np.abs(price_errors / true_prices)) * 100

last_close = all_last_closes[0]                      
pred_logret = all_preds[0]                           
pred_price   = last_close * np.exp(pred_logret)      

print(f"Predicted next-day price for your sample window: ${pred_price:.2f}")

print(f"Price‐level MAPE: {price_mape:.2f}%")

Predicted next-day price for your sample window: $141.87
Price‐level MAPE: 1.92%
